# Canada's Section 338 export exposure

I use this notebook as the walkthrough for the research package. It mirrors `Q2a.py` and `Q2b.py` so you can follow the framework without reading every helper.

## Research questions

**Part (a).** Estimate Canada's export exposure to U.S. Section 338 tariffs as a lower/upper range and show the provincial/territorial distribution.

**Part (b).** For Alberta, report major exposed HS4 products (value, intensity, contribution). Compare Alberta with Saskatchewan, Ontario, Quebec, and British Columbia on value, intensity, composition, concentration (HHI), and an LFS employment proxy.

**Default mode.** `REFIT = False` loads saved tables. You can set `REFIT = True` to re-run the scripts.


## Framework at a glance

```text
Part (a)  Yale S338 HTS8
          UB = HS6 any-match on CIMT HS6 → U.S.
          LB = exact HS8 match on CIMT HS8 → U.S.
          → national range + province/territory distribution

Part (b)  Same screens, roll to HS4
          → Alberta top 10 (E, I, C)
          → peers AB, SK, ON, QC, BC
          → Jaccard + cosine, HHI, LFS employment proxy
```


## Exposure metrics

- **Upper bound:** any listed U.S. HTS8 truncates to HS6 → full Canadian HS6→U.S. value.
- **Lower bound:** Canadian HS8 equals a listed U.S. HTS8.
- **Intensity** \(I_{p,k}\) = exposed U.S. exports in HS4 \(k\) ÷ all U.S. exports in HS4 \(k\).
- **Contribution** \(C_{p,k}\) = exposed HS4 \(k\) ÷ province total S338 exposure.

See `methodology/Q2a_methodology.md` and `methodology/Q2b_methodology.md`.


In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path('.').resolve()
OUT_A = ROOT / 'outputs' / 'q2a'
OUT_B = ROOT / 'outputs' / 'q2b'

# False = read last run (preferred for Git). True = call Q2a.py / Q2b.py.
REFIT = False

if REFIT:
    subprocess.run([sys.executable, 'Q2a.py'], cwd=ROOT, check=True)
    subprocess.run([sys.executable, 'Q2b.py'], cwd=ROOT, check=True)


## Part (a) – National and provincial exposure range


In [ ]:
national = pd.read_csv(OUT_A / 'q2a_national_exposure.csv')
by_province = pd.read_csv(OUT_A / 'q2a_exposure_by_province.csv')
by_program = pd.read_csv(OUT_A / 'q2a_exposure_by_program.csv')

display(national)
display(by_province)
display(by_program)
display(Image(filename=str(OUT_A / 'q2a_exposure_by_province.png')))


## Part (b) – Alberta products and peer comparison


In [ ]:
alberta_top = pd.read_csv(OUT_B / 'q2b_alberta_top_hs4.csv')
province_summary = pd.read_csv(OUT_B / 'q2b_province_summary.csv')
similarity = pd.read_csv(OUT_B / 'q2b_composition_similarity.csv')
hhi = pd.read_csv(OUT_B / 'q2b_concentration_hhi.csv')
employment = pd.read_csv(OUT_B / 'q2b_employment_risk.csv')
comparator_top = pd.read_csv(OUT_B / 'q2b_comparator_top_hs4.csv')

display(province_summary)
display(alberta_top)
display(similarity)
display(hhi)
display(employment)
display(Image(filename=str(OUT_B / 'q2b_alberta_top_hs4.png')))
display(Image(filename=str(OUT_B / 'q2b_province_intensity.png')))
display(Image(filename=str(OUT_B / 'q2b_composition_similarity.png')))
display(Image(filename=str(OUT_B / 'q2b_concentration_hhi.png')))
display(Image(filename=str(OUT_B / 'q2b_employment_risk.png')))


## Comparator top HS4 tables

Saskatchewan, Ontario, Quebec, and British Columbia top-10 exposed HS4 products.


In [ ]:
for code in ['SK', 'ON', 'QC', 'BC']:
    print(code)
    display(comparator_top.loc[comparator_top['province_code'].eq(code)].head(10))


## Reading the comparison

- **Range:** National exposure is about $10B (LB) to $33B (UB). Report the range, not a single point.
- **Value vs intensity:** Alberta's absolute UB exposure is modest; intensity is low (~1%), similar to Saskatchewan, because energy-heavy U.S. shipments sit mostly outside the list.
- **Composition:** Cosine similarity uses the full exposure-share vector; Jaccard only compares top-10 membership. They can diverge (e.g. AB–SK).
- **HHI:** Higher means fewer products dominate the exposed basket (Saskatchewan is the most concentrated among these peers).
- **Employment proxy:** Exposed dollars per goods-sector employee is relative context only. It is not a layoff forecast.

Details: `methodology/Q2a_methodology.md`, `methodology/Q2b_methodology.md`.
